# **Day 20: More Dictionary Methods and a Mini Project**

This is the last day of Phase 1 (Basics). Today we learn the remaining dictionary methods, **nested dictionaries**, and build a **Durga Puja pandal hopping planner**.

In [1]:
print([m for m in dir(dict) if not m.startswith("_")])

['clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']


## **update, setdefault, fromkeys and popitem**
- **update():** Adds new pairs and changes existing ones.
- **setdefault():** Adds a key only if it is **not already** there.
- **fromkeys():** Makes a new dictionary with the same value for many keys.
- **popitem():** Removes the last added pair.

In [2]:
fare = {"bus": 10, "auto": 15}
fare.update({"metro": 20, "auto": 20})
print(fare)

fare.setdefault("tram", 7)      # Added, because "tram" was missing
fare.setdefault("bus", 99)      # Not changed, because "bus" already exists
print(fare)

print(dict.fromkeys(["Kolkata", "Howrah", "Hooghly"], 0))
print(fare.popitem())

{'bus': 10, 'auto': 20, 'metro': 20}
{'bus': 10, 'auto': 20, 'metro': 20, 'tram': 7}
{'Kolkata': 0, 'Howrah': 0, 'Hooghly': 0}
('tram', 7)


## **Merging Dictionaries with |**

In [3]:
a = {"x": 1, "y": 2}
b = {"y": 20, "z": 30}
print(a | b)       # If a key is in both, the value from b wins

{'x': 1, 'y': 20, 'z': 30}


## **Nested Dictionaries**
A **nested dictionary** is a dictionary inside another dictionary. It is useful for storing detailed information.

In [4]:
wb = {
    "Kolkata": {"division": "Presidency", "pin": 700001, "rivers": ["Hooghly"]},
    "Paschim Medinipur": {"division": "Medinipur", "hq": "Midnapore", "rivers": ["Kangsabati", "Subarnarekha"]},
    "Darjeeling": {"division": "Jalpaiguri", "hq": "Darjeeling", "rivers": ["Teesta", "Mahananda"]},
}

print(wb["Darjeeling"]["rivers"][0])

for district, info in wb.items():
    print(f"{district}: {', '.join(info['rivers'])}")

# Safe reading of a key that may not exist
print(wb.get("Nadia", {}).get("hq", "No data"))

Teesta
Kolkata: Hooghly
Paschim Medinipur: Kangsabati, Subarnarekha
Darjeeling: Teesta, Mahananda
No data


## **Example from GIS: A Point Feature with Attributes**
In GIS, every map object (called a **feature**) has two parts:
1. **Geometry:** Where it is (its coordinates).
2. **Attributes (properties):** Information about it (name, type, population and so on).

A nested dictionary is a natural way to store this. In fact, the popular **GeoJSON** format (Day 36) looks exactly like this.

In [5]:
feature = {
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [87.3199, 22.4257]},
    "properties": {"name": "Midnapore", "district": "Paschim Medinipur", "pin": 721101},
}

lon, lat = feature["geometry"]["coordinates"]
print("Name:", feature["properties"]["name"])
print("Location:", lat, "N,", lon, "E")
print("Geometry type:", feature["geometry"]["type"])

Name: Midnapore
Location: 22.4257 N, 87.3199 E
Geometry type: Point


- In GeoJSON, coordinates are written as **[longitude, latitude]** (x first, then y). This is a common source of mistakes, so it is good to remember.

## **Sorting a Dictionary by Value**

In [6]:
footfall = {"Bagbazar": 5.2, "Santosh Mitra Square": 9.8, "Suruchi Sangha": 7.1, "Kumartuli Park": 6.4}
# Made-up visitor numbers in lakh

for name, lakh in sorted(footfall.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{name:<22}{lakh:>5} lakh")

print("Most crowded:", max(footfall, key=footfall.get))

Santosh Mitra Square    9.8 lakh
Suruchi Sangha          7.1 lakh
Kumartuli Park          6.4 lakh
Bagbazar                5.2 lakh
Most crowded: Santosh Mitra Square


## **Mini Project: Pandal Hopping Planner**
**The problem:** We have a limited number of hours on Ashtami night. Which pandals can we visit?

**The plan:**
1. Keep each pandal's area and the time needed to see it.
2. Group the pandals by area (North, Central, South), so we do not keep crossing the city.
3. Visit the areas in order, and add pandals while time is left.
4. Add 30 minutes of travel time when moving to a new area.

In [7]:
pandals = {
    "Bagbazar Sarbojanin": {"area": "north", "mins": 40},
    "Kumartuli Park": {"area": "north", "mins": 35},
    "Ahiritola": {"area": "north", "mins": 30},
    "Mohammed Ali Park": {"area": "central", "mins": 45},
    "College Square": {"area": "central", "mins": 30},
    "Santosh Mitra Square": {"area": "central", "mins": 60},
    "Deshapriya Park": {"area": "south", "mins": 40},
    "Ekdalia Evergreen": {"area": "south", "mins": 35},
    "Suruchi Sangha": {"area": "south", "mins": 45},
    "Chetla Agrani": {"area": "south", "mins": 40},
}
travel_between_areas = 30
available_hours = 6

# Step 1: Group by area
by_area = {}
for name, info in pandals.items():
    by_area.setdefault(info["area"], []).append(name)
print(by_area)

{'north': ['Bagbazar Sarbojanin', 'Kumartuli Park', 'Ahiritola'], 'central': ['Mohammed Ali Park', 'College Square', 'Santosh Mitra Square'], 'south': ['Deshapriya Park', 'Ekdalia Evergreen', 'Suruchi Sangha', 'Chetla Agrani']}


In [8]:
# Step 2: Pick pandals while time is left
time_left = available_hours * 60
plan = []
prev_area = None

for area in ["north", "central", "south"]:
    for name in by_area[area]:
        need = pandals[name]["mins"]
        if prev_area is not None and prev_area != area:
            need += travel_between_areas
        if need <= time_left:
            plan.append(name)
            time_left -= need
            prev_area = area

print(f"We can visit {len(plan)} pandals in {available_hours} hours:")
for i, p in enumerate(plan, 1):
    print(f" {i}. {p} ({pandals[p]['area']})")
print(f"Spare time: {time_left:.0f} minutes for phuchka")

We can visit 7 pandals in 6 hours:
 1. Bagbazar Sarbojanin (north)
 2. Kumartuli Park (north)
 3. Ahiritola (north)
 4. Mohammed Ali Park (central)
 5. College Square (central)
 6. Santosh Mitra Square (central)
 7. Deshapriya Park (south)
Spare time: 20 minutes for phuchka


Let us understand the code above:
- `by_area.setdefault(info["area"], []).append(name)`: If the area is new, it first creates an empty list, and then adds the pandal to it.
- `if need <= time_left`: We add a pandal only if we have enough time left.
- This method is simple: it always takes pandals in order. It may not always find the best possible plan, but it works well enough for Ashtami night.

## **Summary of Phase 1 (Day 1 to Day 20)**
| Topic | Days |
|---|---|
| print, variables, data types, operators | 1 to 3 |
| Strings and string methods | 4 and 5 |
| Input, conditions, match-case | 6 to 9 |
| while and for loops | 10 to 13 |
| Lists, tuples, sets, dictionaries | 15 to 20 |

Next, in Phase 2, we will learn **functions**, **modules** and **working with files**.